In [74]:
import pandas as pd


import pandas as pd

file_path = input("Enter dataset file path: ")
df = pd.read_csv(file_path)

print(df.head())


def detect_columns(df):
    numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    return numerical_cols, categorical_cols


Enter dataset file path: /content/customer (1).csv
   age  gender   review education purchased
0   30  Female  Average    School        No
1   68  Female     Poor        UG        No
2   70  Female     Good        PG        No
3   72  Female     Good        PG        No
4   16  Female  Average        UG        No


In [75]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()


In [76]:
num_cols

['age']

In [77]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


In [78]:
cat_cols

['gender', 'review', 'education', 'purchased']

In [79]:
import re

def detect_categorical_type(df, col):
    """
    Detect whether a categorical column is ordinal or nominal
    using semantic patterns common in Kaggle & real-world datasets.
    """

    values = (
        df[col]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .unique()
    )

    # ---------- ORDINAL WORD GROUPS ----------
    ordinal_sets = [

        # 🔹 Basic intensity
        {"very low", "low", "medium", "high", "very high"},
        {"low", "medium", "high"},

        # 🔹 Quality / Performance
        {"poor", "below average", "average", "good", "very good", "excellent"},
        {"bad", "fair", "good", "very good", "excellent"},

        # 🔹 Skill / Expertise
        {"beginner", "intermediate", "advanced", "expert"},
        {"novice", "competent", "proficient", "expert"},

        # 🔹 Education level
        {"school", "high school", "diploma", "ug", "bachelor", "pg", "master", "phd", "doctorate"},
        {"10th", "12th", "graduate", "postgraduate"},
        {"primary", "secondary", "higher secondary", "tertiary"},

        # 🔹 Job / Experience hierarchy
        {"intern", "junior", "associate", "mid", "senior", "lead", "manager", "director", "vp", "vice president", "c-level"},
        {"trainee", "executive", "senior executive", "manager", "senior manager"},
        {"l1", "l2", "l3", "l4", "l5"},

        # 🔹 Size / Range
        {"very small", "small", "medium", "large", "very large"},
        {"xs", "s", "m", "l", "xl", "xxl"},
        {"tiny", "small", "moderate", "big", "huge"},

        # 🔹 Ratings
        {"1 star", "2 star", "3 star", "4 star", "5 star"},
        {"one", "two", "three", "four", "five"},

        # 🔹 Satisfaction / Agreement
        {"very dissatisfied", "dissatisfied", "neutral", "satisfied", "very satisfied"},
        {"strongly disagree", "disagree", "neutral", "agree", "strongly agree"},

        # 🔹 Priority / Severity
        {"low", "medium", "high", "critical"},
        {"minor", "major", "severe", "critical"},
        {"p1", "p2", "p3", "p4"},

        # 🔹 Risk
        {"low risk", "medium risk", "high risk"},
        {"safe", "warning", "danger"},

        # 🔹 Frequency
        {"never", "rarely", "sometimes", "often", "always"},
        {"rare", "occasional", "frequent", "very frequent"},

        # 🔹 Salary / Income band
        {"entry level", "mid level", "senior level"},
        {"low income", "middle income", "high income"},

        # 🔹 Order / Ranking
        {"first", "second", "third", "fourth", "fifth"},
        {"bronze", "silver", "gold", "platinum", "diamond"},

        # 🔹 Time / Urgency
        {"immediate", "short term", "medium term", "long term"},
        {"early", "on time", "late", "very late"}
    ]

    # ---------- NOMINAL ENTITY WORDS ----------
    nominal_entities = {

        # 🔹 Gender / Identity
        "male", "female", "other", "unknown",

        # 🔹 Geography / Location
        "city", "town", "village", "district",
        "state", "province", "region",
        "country", "nation", "continent",
        "zip", "zipcode", "postal", "pincode",

        # 🔹 Organization / Work
        "company", "organization", "firm",
        "department", "team", "division",
        "branch", "unit",

        # 🔹 Product / Business
        "brand", "product", "model", "variant",
        "category", "subcategory", "type",
        "segment", "class", "series",

        # 🔹 Color / Appearance
        "color", "shade", "tone",

        # 🔹 Job / Profession
        "job", "occupation", "role", "designation",
        "position", "title",

        # 🔹 Industry / Domain
        "industry", "sector", "domain", "field",

        # 🔹 Customer / User
        "customer", "client", "user", "subscriber",
        "account", "profile",

        # 🔹 Identifiers (Always nominal)
        "id", "uuid", "code", "ref", "reference",
        "number", "serial", "registration",

        # 🔹 Status / Labels (non-ordinal)
        "status", "label", "tag", "flag", "indicator",
        "yes", "no", "true", "false",

        # 🔹 Language / Culture
        "language", "nationality", "ethnicity",

        # 🔹 Time-based nominal
        "day", "month", "year", "weekday",

        # 🔹 Platform / Technology
        "device", "platform", "os", "browser",
        "app", "application", "software",

        # 🔹 Marketing / Sales
        "channel", "source", "medium", "campaign",
        "market", "audience",

        # 🔹 Education (Non-ordinal naming)
        "school_name", "college", "university",
        "institute", "board",

        # 🔹 Finance
        "currency", "payment", "transaction",
        "account_type", "plan",

        # 🔹 Misc
        "symbol", "icon", "emoji", "code_type"
    }


    # ---------- RULE 1: Ordinal keyword groups ----------
    for ord_set in ordinal_sets:
        if set(values).issubset(ord_set):
            return "ordinal"

    # ---------- RULE 2: Numeric ranking ----------
    if all(v.isdigit() for v in values):
        return "ordinal"

    # ---------- RULE 3: Level / Stage pattern ----------
    if all(re.fullmatch(r"(level|lvl|stage|grade)\\s*\\d+", v) for v in values):
        return "ordinal"

    # ---------- RULE 4: Experience pattern (1 year, 2 years) ----------
    if all(re.fullmatch(r"\\d+\\s*(year|years|yr|yrs)", v) for v in values):
        return "ordinal"

    # ---------- RULE 5: Grade letters ----------
    if set(values).issubset({"a", "b", "c", "d", "e", "f"}):
        return "ordinal"

    # ---------- RULE 6: Explicit nominal entities ----------
    if any(v in nominal_entities for v in values):
        return "nominal"

    # ---------- SAFE DEFAULT ----------
    return "nominal"

In [80]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
def auto_encode_categorical(df):
    df = df.copy()
    encoded_dfs = []
    encoders = {}

    for col in df.select_dtypes(include="object").columns:
        cat_type = detect_categorical_type(df, col)
        unique_vals = df[col].nunique()

        # -------- ORDINAL --------
        if cat_type == "ordinal":
            encoder = OrdinalEncoder()
            df[col] = encoder.fit_transform(df[[col]])
            encoders[col] = "OrdinalEncoder"

        # -------- NOMINAL (BINARY) --------
        elif unique_vals == 2:
            encoder = LabelEncoder()
            df[col] = encoder.fit_transform(df[col])
            encoders[col] = "LabelEncoder"

        # -------- NOMINAL (MULTI) --------
        else:
            encoder = OneHotEncoder(sparse_output=False, drop="first")
            encoded = encoder.fit_transform(df[[col]])
            encoded_df = pd.DataFrame(
                encoded,
                columns=encoder.get_feature_names_out([col]),
                index=df.index
            )
            df = df.drop(columns=[col])
            df = pd.concat([df, encoded_df], axis=1)
            encoders[col] = "OneHotEncoder"

    return df, encoders


In [81]:
encoded_df, used_encoders = auto_encode_categorical(df)
print(encoded_df.sample(5))



    age  gender  review  education  purchased
26   53       0     2.0        0.0          0
36   34       0     1.0        2.0          1
4    16       0     0.0        2.0          0
18   19       1     1.0        1.0          0
14   15       1     2.0        0.0          1


In [82]:
encoded_df

,age,gender,review,education,purchased
0,30,0,0.0,1.0,0
1,68,0,2.0,2.0,0
2,70,0,1.0,0.0,0
3,72,0,1.0,0.0,0
4,16,0,0.0,2.0,0
5,31,0,0.0,1.0,1
6,18,1,1.0,1.0,0
7,60,0,2.0,1.0,1
8,65,0,0.0,2.0,0
9,74,1,1.0,2.0,1


In [83]:
print(used_encoders)


{'gender': 'LabelEncoder', 'review': 'OrdinalEncoder', 'education': 'OrdinalEncoder', 'purchased': 'LabelEncoder'}


In [58]:
df

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No
5,31,Female,Average,School,Yes
6,18,Male,Good,School,No
7,60,Female,Poor,School,Yes
8,65,Female,Average,UG,No
9,74,Male,Good,UG,Yes
